### Test Results

### MERGING NEIGHBORING MESHES

#### Case 1

In [1]:
import pymesh
from utils import *

In [13]:
mesh = pymesh.load_mesh("../examples/archive/data/two_spheres.ply")

In [14]:
method1_mesh = direct_repair(mesh)
method2_mesh = pymesh.compute_outer_hull(mesh)
method3_mesh = repair_meshlib(path="../examples/archive/data/two_spheres.ply", details=compute_median_edge_length(mesh)/4) # Fix self-intersections by converting to voxels and back.
method4_mesh = repair_contour_original("../examples/archive/SNResult/case1/combined_labels.obj")          # repair_contour() is now depricated
method5_mesh = repair_with_local_remesh(mesh) 

In [5]:
full_evaluation2(mesh, method1_mesh, method2_mesh, method3_mesh, method4_mesh, method5_mesh)

+-------------------------+-------------+-------------+--------------+---------------+---------------+----------------+
| Metric                  |    Original |   PyMeshFix |      MeshFix |       Meshlib |   SurfaceNets |   Local Remesh |
+=========================+=============+=============+==============+===============+===============+================+
| vertices                |  952        |  476        |  917         | 15760         |  34766        |    937         |
+-------------------------+-------------+-------------+--------------+---------------+---------------+----------------+
| faces                   | 1896        |  948        | 1830         | 31516         |  73132        |   1870         |
+-------------------------+-------------+-------------+--------------+---------------+---------------+----------------+
| intersecting face pairs |  119        |    0        |    0         |     0         |      0        |      0         |
+-------------------------+-------------

#### Case 2

In [16]:
mesh2 = pymesh.load_mesh("../examples/archive/data/three_spheres.ply")

In [17]:
method1_mesh2 = direct_repair(mesh2)
method2_mesh2 = pymesh.compute_outer_hull(mesh2)
method3_mesh2 = repair_meshlib(path="../examples/archive/data/three_spheres.ply", details=compute_median_edge_length(mesh2)/4)
method4_mesh2 = repair_contour_original("../examples/archive/SNResult/case2/combined_labels.obj")
method5_mesh2 = repair_with_local_remesh(mesh2)

In [8]:
full_evaluation2(mesh2, method1_mesh2, method2_mesh2, method3_mesh2, method4_mesh2, method5_mesh2)

+-------------------------+-------------+-------------+--------------+---------------+----------------+----------------+
| Metric                  |    Original |   PyMeshFix |      MeshFix |       Meshlib |    SurfaceNets |   Local Remesh |
+=========================+=============+=============+==============+===============+================+================+
| vertices                | 1608        |  536        | 1576         | 26588         |  61786         |   1564         |
+-------------------------+-------------+-------------+--------------+---------------+----------------+----------------+
| faces                   | 3204        | 1068        | 3148         | 53172         | 129624         |   3124         |
+-------------------------+-------------+-------------+--------------+---------------+----------------+----------------+
| intersecting face pairs |  292        |    0        |    0         |     0         |      0         |      0         |
+-------------------------+-----

#### Case 3

In [19]:
mesh3 = pymesh.load_mesh("../examples/archive/data/sphere_cube.ply")

In [20]:
method1_mesh3 = direct_repair(mesh3)
method2_mesh3 = pymesh.compute_outer_hull(mesh3)
method3_mesh3 = repair_meshlib(path="../examples/archive/data/sphere_cube.ply", details=compute_median_edge_length(mesh3)/4)
method4_mesh3 = repair_contour_original("../examples/archive/SNResult/case3/combined_labels.obj")
method5_mesh3 = repair_with_local_remesh(mesh3)

In [11]:
full_evaluation2(mesh3, method1_mesh3, method2_mesh3, method3_mesh3, method4_mesh3, mesh5=method5_mesh3)

+-------------------------+-------------+-------------+--------------+---------------+---------------+----------------+
| Metric                  |    Original |   PyMeshFix |      MeshFix |       Meshlib |   SurfaceNets |   Local Remesh |
+=========================+=============+=============+==============+===============+===============+================+
| vertices                |  760        |  230        |  758         | 11064         |  23773        |    860         |
+-------------------------+-------------+-------------+--------------+---------------+---------------+----------------+
| faces                   | 1512        |  456        | 1512         | 22124         |  48932        |   1716         |
+-------------------------+-------------+-------------+--------------+---------------+---------------+----------------+
| intersecting face pairs |   99        |    0        |    0         |     0         |      0        |      0         |
+-------------------------+-------------

#### Case 4

In [ ]:
mesh4 = pymesh.load_mesh("data/brain.ply")

In [ ]:
method1_mesh4 = direct_repair(mesh4)
method2_mesh4 = pymesh.compute_outer_hull(mesh4)
method3_mesh4 = repair_meshlib(path="data/brain.ply", details=compute_median_edge_length(mesh4)/2)
method4_mesh4 = repair_contour(mesh4, voxel_size=compute_median_edge_length(mesh4)/2)
method5_mesh4 = repair_with_local_remesh(mesh4)

In [ ]:
full_evaluation2(original=mesh4, mesh1=method1_mesh4, mesh2=method2_mesh4, mesh3=method3_mesh4, mesh4=method4_mesh4, mesh5=method5_mesh4)

In [3]:
import pyvista as pv

def paper_two_meshes(mesh1, mesh2, filename1="Original", filename2="Repaired"):
    mesh1 = convert_to_pyvista(mesh1)
    mesh2 = convert_to_pyvista(mesh2)
    plotter = pv.Plotter(shape=(1, 2), notebook=True)

    plotter.subplot(0, 0)
    mesh_actor1 = plotter.add_mesh(
        mesh1,
        color="white",
        show_edges=True,
        edge_color="black",
        label=filename1,
        line_width=0.3
    )
    plotter.add_text(filename1, position="upper_left", font_size=20)

    plotter.subplot(0, 1)
    mesh_actor2 = plotter.add_mesh(
        mesh2,
        color="white",
        show_edges=True,
        edge_color="black",
        label=filename2,
        line_width=0.3
    )
    plotter.add_text(filename2, position="upper_left", font_size=20)

    plotter.link_views()

    def update_clipping_plane(value):
        # Plane for mesh1
        plane1 = pv.Plane(center=(value, 0, 0), direction=(3, 1, 0))
        clipped_mesh1 = mesh1.clip_surface(plane1, invert=False)
        mesh_actor1.mapper.SetInputData(clipped_mesh1)

        # Plane for mesh2
        plane2 = pv.Plane(center=(value, 0, 0), direction=(1, 0, 0))
        clipped_mesh2 = mesh2.clip_surface(plane2, invert=False)
        mesh_actor2.mapper.SetInputData(clipped_mesh2)

        plotter.render()

    plotter.add_slider_widget(
        callback=update_clipping_plane,
        rng=[-2, 2],
        value=0,
        title="Clip Plane Both Meshes",
        pointa=(0.2, 0.1),
        pointb=(0.8, 0.1)
    )

    plotter.add_legend()

    display(plotter.show(jupyter_backend="trame"))

    return plotter

### DETACHING ENCLOSED MESHES

#### Two spheres

In [2]:
import sys
sys.path.append("..")
import fixmesh
mesh = pymesh.load_mesh("../examples/archive/data/shrink2.ply")
sol2_mesh = fixmesh.detach_repair(pymesh_to_trimesh(mesh))
sol2_mesh_p = pymesh.form_mesh(np.asarray(sol2_mesh.vertices), np.asarray(sol2_mesh.triangles))

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Compute normals for better sampling
Sample outer mesh and build KD-tree
Prepare inner vertices and center
Set up RaycastingScene
0 iteration
1 iteration
2 iteration
3 iteration
4 iteration
5 iteration
6 iteration
7 iteration
8 iteration
9 iteration


In [3]:
full_evaluation_before_after(mesh, sol2_mesh_p)

+-------------------------+-------------+---------------+
| Metric                  |    Original |         After |
+=========================+=============+===============+
| vertices                |  736        |  736          |
+-------------------------+-------------+---------------+
| faces                   | 1464        | 1464          |
+-------------------------+-------------+---------------+
| intersecting face pairs |  101        |    0          |
+-------------------------+-------------+---------------+
| volume                  |    5.00062  |    4.88514    |
+-------------------------+-------------+---------------+
| area                    |   17.3776   |   17.0637     |
+-------------------------+-------------+---------------+
| mean aspect ratio       |    1.359    |    1.35906    |
+-------------------------+-------------+---------------+
| mMean condition         |    1.17213  |    1.17231    |
+-------------------------+-------------+---------------+
| mean max ang

#### Head

In [2]:
import sys
sys.path.append("..")
import fixmesh

mesh = pymesh.load_mesh("../examples/archive/data/whole_head2.ply")
combined_mesh, i, j = find_intersecting_pair_by_self_intersection(mesh)
fixed_mesh = fixmesh.detach_repair(pymesh_to_trimesh(combined_mesh), contact_threshold=0.4, num_sample_points=200000)
fixed_mesh_p = pymesh.form_mesh(np.asarray(fixed_mesh.vertices), np.asarray(fixed_mesh.triangles))

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Found self-intersection between submesh 0 and 2
Compute normals for better sampling
Sample outer mesh and build KD-tree
Prepare inner vertices and center
Set up RaycastingScene
0 iteration
1 iteration
2 iteration
3 iteration
4 iteration
5 iteration
6 iteration
7 iteration
8 iteration
9 iteration


In [3]:
len(pymesh.detect_self_intersection(fixed_mesh_p))

0

In [5]:
full_evaluation_before_after(combined_mesh, fixed_mesh_p)

+-------------------------+---------------+----------------+
| Metric                  |      Original |          After |
+=========================+===============+================+
| vertices                | 407606        | 407606         |
+-------------------------+---------------+----------------+
| faces                   | 815224        | 815224         |
+-------------------------+---------------+----------------+
| intersecting face pairs |   6407        |      0         |
+-------------------------+---------------+----------------+
| volume                  | 678788        | 687419         |
+-------------------------+---------------+----------------+
| area                    | 175553        | 175334         |
+-------------------------+---------------+----------------+
| mean aspect ratio       |      1.25318  |      1.25355   |
+-------------------------+---------------+----------------+
| mMean condition         |      1.10248  |      1.10273   |
+-----------------------

### DETACHING NEIGHBORING MESHES

#### Two Spheres

In [2]:
import sys
sys.path.append("..")
import fixmesh
mesh = pymesh.load_mesh("../examples/archive/data/two_spheres.ply")
sol3_mesh = fixmesh.cut_repair(mesh)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Found 4 components after cut. Keeping 2 largest.


In [3]:
full_evaluation_before_after(mesh, sol3_mesh)

+-------------------------+-------------+--------------+
| Metric                  |    Original |        After |
+=========================+=============+==============+
| vertices                |  952        |  882         |
+-------------------------+-------------+--------------+
| faces                   | 1896        | 1756         |
+-------------------------+-------------+--------------+
| intersecting face pairs |  119        |    0         |
+-------------------------+-------------+--------------+
| volume                  |    7.07818  |    6.50881   |
+-------------------------+-------------+--------------+
| area                    |   22.5682   |   21.8351    |
+-------------------------+-------------+--------------+
| mean aspect ratio       |    1.44276  |    1.42534   |
+-------------------------+-------------+--------------+
| mMean condition         |    1.24015  |    1.2261    |
+-------------------------+-------------+--------------+
| mean max angle          |   8

#### Bended Cylinder

In [6]:
mesh2 = pymesh.load_mesh("../examples/archive/data/bend_cylinder.ply")
sol3_mesh2 = fixmesh.cut_repair(mesh2)

Found 4 components after cut. Keeping 1 largest.


In [7]:
full_evaluation_before_after(mesh2, sol3_mesh2)

+-------------------------+--------------+----------------+
| Metric                  |     Original |          After |
+=========================+==============+================+
| vertices                | 2520         | 2288           |
+-------------------------+--------------+----------------+
| faces                   | 4604         | 4572           |
+-------------------------+--------------+----------------+
| intersecting face pairs | 1845         |    0           |
+-------------------------+--------------+----------------+
| volume                  |    0.0611926 |    0.060941    |
+-------------------------+--------------+----------------+
| area                    |    1.84035   |    1.83772     |
+-------------------------+--------------+----------------+
| mean aspect ratio       |    1.98231   |    1.86178     |
+-------------------------+--------------+----------------+
| mMean condition         |    1.73438   |    1.5858      |
+-------------------------+-------------